# myoSuperMimic (PR #573) and its InstantTrack changes on a Colab GPU: install, test, time, A/B, troubleshoot

Runtime → Change runtime type → **GPU** (T4 works; L4/A100 are faster). Run the cells in order.
The last cell prints a `PASTE_BACK` block: copy it into the chat.

What this checks:
1. the branch installs with mjlab 1.6.0 / mujoco-warp 3.11.0 (mujoco 3.11.x, as mjlab requires);
2. the PR's own tests pass on GPU;
3. a timed SuperMimic run on the KIT walking clip: simulator steps/s, seconds per world-model + actor update, GPU memory, reward curve;
4. if something fails, the troubleshooting cell retries with the usual fallbacks (no `torch.compile`, smaller batches).

`BRANCH = "pr573"` runs the PR as is (public). `BRANCH = "feat/supermimic-instanttrack"` (MyoHub/myosuite, MyoFullBody) adds the InstantTrack changes.
The A/B cell then trains the PR baseline and `--instanttrack` for the same wall time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print(sys.version)

In [ ]:
BRANCH = "feat/supermimic-instanttrack"   # or "pr573"
import os, subprocess
os.chdir("/content")
url = "https://github.com/MyoHub/myosuite.git"
if not os.path.isdir("myosuite"):
    subprocess.run(["git", "clone", "-q", url, "myosuite"], check=True)
os.chdir("/content/myosuite")
if BRANCH == "pr573":
    subprocess.run(["git", "fetch", "-q", "origin", "pull/573/head:pr573"], check=True)
else:
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "-q", BRANCH if BRANCH == "pr573" else "FETCH_HEAD"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# mjlab 1.6.0 pins mujoco ~= 3.11 and mujoco-warp ~= 3.11: let pip pick mujoco.
# [musclemimic]: myoMimicFullbody-v0 builds MyoFullBody from the musclemimic_models MJCF (354 muscles).
!pip -q install -e ".[mjlab,musclemimic]" "mjlab==1.6.0" "mujoco-warp==3.11.0" "warp-lang>=1.14" tyro pytest 2>&1 | tail -3
!pip check 2>&1 | grep -iE "mujoco|mjlab|warp|torch" || echo "pip check: no conflicts for mujoco/mjlab/warp/torch"
!python -c "import torch, warp, mujoco, mujoco_warp, mjlab; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.version.cuda); print('warp', warp.__version__, 'mujoco', mujoco.__version__)"

If the import line fails after the install, use Runtime → Restart session and run the previous cell again (Colab sometimes keeps an older torch loaded).

In [ ]:
%%bash
cd /content/myosuite
python -m pytest -q myosuite/tests/test_supermimic*.py 2>&1 | tail -15

In [ ]:
from huggingface_hub import hf_hub_download
CLIP = hf_hub_download("amathislab/musclemimic-retargeted",
                       "MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz", repo_type="dataset")
print(CLIP)

## Timed run

`HOURS` bounds the run (0.25 h gives timing and the first evaluations; the PR's best result needed ~9 h on an RTX 5090).
On a T4 (16 GB) start with 256 envs; on L4/A100 use 512 as in the PR.

In [ ]:
import subprocess, threading, time, torch, os
HOURS = 0.25
NUM_ENVS = 512 if torch.cuda.get_device_properties(0).total_memory > 20e9 else 256
OUT = "/content/runs/sm"
EXTRA = []          # e.g. ["--no-compile"] or ["--imag-batch", "256"] (see troubleshooting)
cmd = ["python", "scripts/supermimic/train.py", "--clip", CLIP, "--out", OUT,
       "--num-envs", str(NUM_ENVS), "--horizon", "64", "--imag-batch", "512", "--wm-batches", "4",
       "--seed", "0", "--max-hours", str(HOURS), "--eval-every", "250", "--eval-envs", "64",
       "--save-every", "100000"] + EXTRA
print(" ".join(cmd))
peak = [0]
def watch():
    while proc.poll() is None:
        q = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.split(",")
        try: peak[0] = max(peak[0], int(q[0]))
        except Exception: pass
        time.sleep(10)
t0 = time.time()
with open("/content/train.log", "w") as f:
    proc = subprocess.Popen(cmd, cwd="/content/myosuite", stdout=f, stderr=subprocess.STDOUT)
    threading.Thread(target=watch, daemon=True).start()
    proc.wait()
WALL = time.time() - t0
print("exit", proc.returncode, "wall min", round(WALL / 60, 1), "peak GPU MB", peak[0])
!grep -v "^Module" /content/train.log | tail -20

In [ ]:
import json, matplotlib.pyplot as plt
recs = [json.loads(l) for l in open(f"{OUT}/log.jsonl")] if os.path.exists(f"{OUT}/log.jsonl") else []
prog = [r for r in recs if not r.get("final")]
for r in prog[-5:]:
    n = 50  # log records are every 50 updates
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()
           if k in ("updates", "sim_steps", "hours", "train_ep_len", "t_sim", "t_update", "wm_loss", "eval_reward_per_step", "eval_leg_rmse_deg")})
if prog:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot([r["sim_steps"] / 1e6 for r in prog], [r["train_ep_len"] for r in prog]); ax[0].set_xlabel("sim steps (M)"); ax[0].set_ylabel("train episode length")
    ev = [r for r in recs if "eval_reward_per_step" in r]
    ax[1].plot([r["sim_steps"] / 1e6 for r in ev], [r["eval_reward_per_step"] for r in ev], "o-"); ax[1].set_xlabel("sim steps (M)"); ax[1].set_ylabel("eval reward / step (max 5)")
    plt.tight_layout(); plt.show()

## A/B: PR baseline vs `--instanttrack` vs `--instanttrack --act-grad-clip 3` (NablaMimic's gradient clip)

Same clip, envs, horizon, seed and wall time (`AB_HOURS` each, run one after the other). Compares the evaluation reward per step
(max 5), leg joint-angle RMSE and episode length of the final and averaged actors, against simulator steps and wall time.

In [ ]:
AB_HOURS = 0.5
# arms run one after the other; the third adds NablaMimic's per-sample action-gradient clip on top of InstantTrack
ARMS = [("baseline", []), ("instanttrack", ["--instanttrack"]),
        ("instanttrack_nabla", ["--instanttrack", "--act-grad-clip", "3"])]
runs = {}
for tag, extra in ARMS:
    out = f"/content/runs/ab_{tag}"
    c = ["python", "scripts/supermimic/train.py", "--clip", CLIP, "--out", out, "--num-envs", str(NUM_ENVS),
         "--horizon", "64", "--imag-batch", "512", "--wm-batches", "4", "--seed", "0", "--max-hours", str(AB_HOURS),
         "--eval-every", "250", "--eval-envs", "64", "--save-every", "100000"] + extra
    t0 = time.time()
    with open(f"/content/ab_{tag}.log", "w") as f:
        rc = subprocess.run(c, cwd="/content/myosuite", stdout=f, stderr=subprocess.STDOUT).returncode
    recs = [json.loads(l) for l in open(f"{out}/log.jsonl")] if os.path.exists(f"{out}/log.jsonl") else []
    runs[tag] = {"exit": rc, "wall_min": round((time.time() - t0) / 60, 1), "recs": recs}
    print(tag, "exit", rc, "wall min", runs[tag]["wall_min"], "records", len(recs))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for tag, r in runs.items():
    ev = [x for x in r["recs"] if "eval_reward_per_step" in x]
    ax[0].plot([x["hours"] * 60 for x in ev], [x["eval_reward_per_step"] for x in ev], "o-", label=tag)
    ax[1].plot([x["sim_steps"] / 1e6 for x in ev], [x["eval_reward_per_step"] for x in ev], "o-", label=tag)
ax[0].set_xlabel("training minutes"); ax[1].set_xlabel("sim steps (M)")
for a in ax: a.set_ylabel("eval reward / step (max 5)"); a.legend()
plt.tight_layout(); plt.show()
AB = {}
for tag, r in runs.items():
    fin = [x for x in r["recs"] if x.get("final")]
    prog = [x for x in r["recs"] if not x.get("final")]
    AB[tag] = {"exit": r["exit"], "wall_min": r["wall_min"],
               "updates": prog[-1]["updates"] if prog else None, "sim_steps": prog[-1]["sim_steps"] if prog else None,
               **({k: round(fin[0][k], 3) for k in ("eval_reward_per_step", "eval_ema_reward_per_step", "eval_leg_rmse_deg",
                                                    "eval_ema_leg_rmse_deg", "eval_mean_episode_len") if k in fin[0]} if fin else {})}
print("PASTE_BACK_AB " + json.dumps(AB))

## Troubleshooting (run only if the timed run failed or is very slow)

Classifies the failure from `train.log` and reruns 5 minutes with the matching fallback.

In [ ]:
log = open("/content/train.log").read()
fix = []
import re
missing = re.findall(r"(?:ModuleNotFoundError: No module named|ImportError: )'?([\w.]+)", log)
if missing:
    print("missing package (re-run the install cell; it installs [mjlab,musclemimic]):", sorted(set(missing)))
if "OutOfMemory" in log or "out of memory" in log:      fix = ["--num-envs", "128", "--imag-batch", "256"]
elif "torch._dynamo" in log or "InductorError" in log or "BackendCompilerFailed" in log: fix = ["--no-compile"]
elif "CUDA error" in log or "cudaError" in log: fix = ["--no-compile", "--no-tf32"]
print("detected fix:", fix or "none (look at the traceback below)")
print("\n".join([l for l in log.splitlines() if not l.startswith("Module")][-40:]))
if fix:
    retry = list(cmd)
    retry[retry.index("--out") + 1] = OUT + "_fix"
    retry[retry.index("--max-hours") + 1] = "0.08"
    retry += fix  # a later flag overrides the earlier one (argparse semantics)
    r = subprocess.run(retry, cwd="/content/myosuite", capture_output=True, text=True)
    print("retry exit", r.returncode); print("\n".join(r.stdout.splitlines()[-15:] + r.stderr.splitlines()[-15:]))

In [ ]:
import platform
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
tail = [l for l in open("/content/train.log").read().splitlines() if not l.startswith("Module")][-12:]
s = {"branch": BRANCH, "gpu": gpu, "python": platform.python_version(), "torch": torch.__version__, "num_envs": NUM_ENVS, "extra": EXTRA,
     "exit": proc.returncode, "wall_min": round(WALL / 60, 1), "peak_gpu_mb": peak[0]}
if prog:
    last = prog[-1]
    s.update({"updates": last["updates"], "sim_steps": last["sim_steps"],
              "s_per_update": round(sum(r["t_update"] for r in prog) / (50 * len(prog)), 3),
              "sim_steps_per_s": round(sum(50 / 0.4 * NUM_ENVS for r in prog) / max(sum(r["t_sim"] for r in prog), 1e-9), 1),
              "train_ep_len": round(last["train_ep_len"], 1)})
fin = [r for r in recs if r.get("final")]
if fin: s.update({k: round(fin[0][k], 3) for k in ("eval_reward_per_step", "eval_ema_reward_per_step", "eval_leg_rmse_deg", "eval_mean_episode_len") if k in fin[0]})
print("PASTE_BACK " + json.dumps(s))
if proc.returncode: print("LOG_TAIL\n" + "\n".join(tail))